# Pakistani-Context NER Model Fine-Tuning (Colab / Kaggle GPU)
**TechFest NER Challenge: Dory's Data Dive (Round 2)**

This notebook trains `xlm-roberta-base` on the verified Pakistani-Context NER dataset.
- 20 exact entity classes
- Reads verified `v2_frozen/` (if present) or `v2/` dataset
- First-subword alignment with word_ids() (-100 masking)
- Class-weighted cross-entropy loss for rare class resilience
- FP16 mixed precision on T4 GPU
- Automatic checkpoint and config bundle saving to Google Drive

In [ ]:
# 0. Clone repository and navigate to project
# NOTE: If your repo is Private, either make it Public on GitHub (Settings -> Change visibility -> Public)
# or clone using a personal access token: https://<TOKEN>@github.com/Muhammad-Hassan06/AI-Odyssey.git
import os, sys
if not os.path.exists('AI-Odyssey'):
    !git clone https://github.com/Muhammad-Hassan06/AI-Odyssey.git
%cd AI-Odyssey/ner-pk
!git pull origin main


In [ ]:
# 1. Verify Dataset Files & Determine DATA_DIR
import os, hashlib

if os.path.exists('v2_frozen/train.jsonl'):
    DATA_DIR = 'v2_frozen'
elif os.path.exists('v2/train.jsonl'):
    DATA_DIR = 'v2'
else:
    print('❌ ERROR: Neither v2_frozen/ nor v2/ was found in the cloned repository!')
    print('Run this in your local repo, then commit and push:')
    print('  git add v2/ v2_frozen/ -f')
    print('  git commit -m "fix: track v2 datasets for training"')
    print('  git push origin main')
    raise FileNotFoundError('Dataset folder not found.')

print(f'✅ Selected DATA_DIR: {DATA_DIR}')

for split in ['train.jsonl', 'dev.jsonl', 'benchmark.jsonl']:
    p = os.path.join(DATA_DIR, split)
    if os.path.exists(p):
        with open(p, 'rb') as f:
            content = f.read()
            h = hashlib.sha256(content).hexdigest()[:16]
            lines = content.count(b'\n')
            print(f'  - {split}: {lines} lines, sha256={h}...')
    else:
        print(f'  - {split}: missing')


In [ ]:
# 2. Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: CUDA GPU not detected! Go to Runtime -> Change runtime type -> T4 GPU")


In [ ]:
# 3. Upgrade pip tools and install missing packages
# Note: We do NOT downgrade torch, transformers, or accelerate.
!pip install --upgrade -q pip setuptools wheel
!pip install -q seqeval pyyaml

import transformers, accelerate, torch, seqeval
print('Installed environment package versions:')
print(f'  - torch: {torch.__version__}')
print(f'  - transformers: {transformers.__version__}')
print(f'  - accelerate: {accelerate.__version__}')
print(f'  - seqeval: {seqeval.__version__}')


In [ ]:
# 4. Mount Google Drive for persistent checkpoint saving
from google.colab import drive
drive.mount('/content/drive')

# Set output directory on Drive
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/ner_pk_models/best_ner_model_v2"
!mkdir -p {DRIVE_OUTPUT_DIR}
print(f"Target Drive Output Directory: {DRIVE_OUTPUT_DIR}")


In [ ]:
# 5. Launch Training on V2 dataset
!python src/train.py \
    --model_name "xlm-roberta-base" \
    --train_file "{DATA_DIR}/train.jsonl" \
    --dev_file "{DATA_DIR}/dev.jsonl" \
    --output_dir "{DRIVE_OUTPUT_DIR}" \
    --batch_size 16 \
    --epochs 5 \
    --lr 3e-5 \
    --max_length 128 \
    --use_class_weights \
    --fp16 \
    --seed 42


In [ ]:
# 6. Package Model and Config for Download & Local Evaluation
import os, zipfile, glob

zip_path = "/content/drive/MyDrive/ner_pk_models/best_ner_model_v2.zip"
print(f"Creating deployment zip bundle at: {zip_path}")

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(DRIVE_OUTPUT_DIR):
        for f in files:
            if 'checkpoint-' in root:  # Skip intermediate checkpoints
                continue
            fp = os.path.join(root, f)
            arcname = os.path.relpath(fp, DRIVE_OUTPUT_DIR)
            zipf.write(fp, arcname)

print("✅ Done! Download best_ner_model_v2.zip, unzip into models/final/, and run local evaluate.py")
